# 88. Merge Sorted Array

[Problem](https://leetcode.com/problems/merge-sorted-array/) · difficulty: easy

The answer is written into `nums1`, so the space to write into is the space still holding unread
input. This notebook is about that conflict: the two approaches here sidestep it, the follow-up's
answer resolves it, and the fastest program of the three is the one with the worst bound.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0088-merge-sorted-array'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions

solutions = load_solutions(PROBLEM)
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Handles the overwrite by |
|---|---|---|---|
| `SolutionConcatSort` | O((m+n) log(m+n)) | O(m+n) | not merging at all |
| `SolutionForwardInsert` | O(m·n) | O(m) | shifting the tail out of the way |
| `SolutionBackwardTwoPointer` | O(m+n) | O(1) | writing from the end, where the space is |


## Why writing forward is the problem

Merging two sorted arrays is easy when the output is somewhere else. Here the output *is*
`nums1`, and the first write already lands on an element that has not been read yet.

The trace below shows the collision on the statement's own example: writing the merged value at
position 1 would destroy `nums1[1] = 2`, which is still needed.


In [ ]:
def naive_forward(nums1, m, nums2, n):
    """The obvious merge, writing into nums1 from the front. Loses data."""
    i = j = write = 0
    while i < m and j < n:
        if nums1[i] <= nums2[j]:
            nums1[write] = nums1[i]
            i += 1
        else:
            print(f'  writing {nums2[j]} at {write} destroys nums1[{write}]={nums1[write]}, still unread')
            nums1[write] = nums2[j]
            j += 1
        write += 1
    return nums1

nums1, m, nums2, n = [1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3
print('merging', nums1[:m], 'with', nums2)
print('result :', naive_forward(list(nums1), m, list(nums2), n), ' expected [1, 2, 2, 3, 5, 6]')


## The fix: write from the end

Compare the two **largest** unread values and put the winner in the last free slot. The write
cursor starts at `m + n - 1` and the read cursors at `m - 1` and `n - 1`, so the write position
is always at or ahead of both — it can never land on something unread.

That is `SolutionBackwardTwoPointer`. The cell below traces the same loop step by step; the
measurements further down use the class itself.


In [ ]:
def trace_backward(nums1, m, nums2, n):
    """SolutionBackwardTwoPointer's loop, narrated. The class itself is timed below."""
    write = m + n - 1
    left, right = m - 1, n - 1
    while right >= 0:
        if left >= 0 and nums1[left] > nums2[right]:
            print(f'  nums1[{left}]={nums1[left]} wins -> slot {write}   {nums1}')
            nums1[write] = nums1[left]
            left -= 1
        else:
            print(f'  nums2[{right}]={nums2[right]} wins -> slot {write}   {nums1}')
            nums1[write] = nums2[right]
            right -= 1
        write -= 1
    return nums1

print('merging [1, 2, 3] with [2, 5, 6]')
print('result :', trace_backward([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3))


Notice that the write cursor never overtakes `i`: every slot it touches is either padding or an
element already copied. That is the invariant the forward version cannot have.


## What it actually costs

All three at the constraint limit, `m = n = 100`, on the shapes that stress them differently.
`nums2 all smaller` is the worst case for insertion — every value shifts the entire prefix.


In [ ]:
import timeit

def micros(fn, nums1, m, nums2, n):
    base = list(nums1)
    runs = min(timeit.repeat(lambda: fn(list(base), m, list(nums2), n), number=200, repeat=5))
    return runs / 200 * 1e6

SHAPES = {
    'nums2 all larger': (list(range(0, 100)) + [0] * 100, 100, list(range(100, 200)), 100),
    'nums2 all smaller': (list(range(100, 200)) + [0] * 100, 100, list(range(0, 100)), 100),
    'interleaved': (list(range(0, 200, 2)) + [0] * 100, 100, list(range(1, 200, 2)), 100),
}

variants = {s.__name__: s().merge for s in solutions}

print(f"{'approach':<26}" + ''.join(f'{name:>22}' for name in SHAPES))
for name, fn in variants.items():
    row = ''.join(f'{micros(fn, *shape):19.1f} us' for shape in SHAPES.values())
    print(f'{name:<26}{row}')


**The optimal algorithm is not the fastest program.** `SolutionConcatSort` has the worst bound of
the three and wins every shape, because its loop runs in C while the other two interpret Python
bytecode per element. Timsort also detects the two sorted runs and merges them, so the log factor
is mostly theoretical on this input.

The asymptotics only take over at sizes the constraints forbid — at `n ≤ 200` the judge reports
0 ms for all of them.


## The bug this problem is built to punish

`nums1 = nums1[:m]` looks like it trims the array. It builds a *new list* and points the local
name at it, so every later mutation is invisible to the caller — the function computes the right
answer and stores it nowhere.


In [ ]:
def rebinds(nums1, m, nums2, n):
    nums1 = nums1[:m]      # new list; caller's object untouched
    nums1[m:] = nums2
    nums1.sort()


def mutates(nums1, m, nums2, n):
    del nums1[m:]          # same object, truncated in place
    nums1[m:] = nums2
    nums1.sort()


for fn in (rebinds, mutates):
    nums1 = [1, 2, 3, 0, 0, 0]
    fn(nums1, 3, [2, 5, 6], 3)
    print(f'{fn.__name__:<10} caller sees {nums1}')


## Takeaway

- When the output shares storage with the input, direction is the whole design. Writing from the
  end turns an impossible overwrite into a free O(1) merge.
- `x = ...` rebinds, `x[:] = ...` / `del x[...]` / `x.sort()` mutate. A function that rebinds a
  parameter cannot return anything to its caller through it.
- A worse bound can be the faster program at bounded sizes. `list.sort` in C beats a hand-written
  O(m+n) Python loop here, and the judge's 0 ms cannot tell any of them apart.
